[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/main/NuevoPipeline/ETL.ipynb)

# 🔬 Nuevo Pipeline ETL: Extracción Pura de Señales de Vibración (Paderborn 64 kHz)
### Descarga y Ventaneado Crudo de Motores Sanos (Buenos) y con Falla (Malos) — Listo para Tratamiento

---

## 🎯 Objetivo de este Notebook
Este notebook descarga y construye el **dataset físico base 100% puro**, extrayendo la señal del canal de **vibraciones mecánicas (`vibration_1`)** a 64 kHz tanto de motores sanos como de motores con fallas reales por fatiga, garantizando **más sanos que fallas**.

* **Cero transformaciones destructivas:** NO aplica `log1p`, NO aplica `clipping`, NO aplica ningún escalador (`RobustScaler`/`StandardScaler`) y NO normaliza $L_2$.
* **Almacenamiento dual:**
  1. `paderborn_vibracion_raw.csv`: Formato tabular directo con condiciones operativas, métricas temporales de choque (RMS, Kurtosis, Skewness, Crest Factor), FFT cruda y etiquetas.
  2. `paderborn_vibracion_raw.npz`: Archivo compacto de alta velocidad que almacena las **ventanas en el tiempo completas (2048 muestras)** y el **espectro FFT completo (1025 bins hasta 32,000 Hz)** para que el siguiente agente pueda diseñar cualquier filtro, subbandas o envolvente de Hilbert sin tener que volver a descargar gigabytes desde Zenodo.
* **Destino en Google Drive:** Los artefactos quedan guardados en su propia carpeta principal e independiente:
  `MyDrive/Paderborn_QML/NuevoPipeline/`

In [ ]:
# ==============================================================================
# CELDA 1: ENTORNO, RUTAS Y CONFIGURACIÓN DEL NUEVO PIPELINE
# ==============================================================================
import os
import sys
import shutil
import urllib.request
import subprocess
import numpy as np
import pandas as pd
import scipy.io
from scipy.stats import kurtosis, skew

# Dependencias para descompresión RAR
try:
    import rarfile
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rarfile"])
    import rarfile

if sys.platform.startswith("linux") and not shutil.which("unrar"):
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)

# Detección Colab vs Local
EN_COLAB = os.path.exists("/content") and ("google.colab" in sys.modules or os.path.exists("/content/drive"))

# Carpetas en Google Drive
CARPETA_RAIZ_DRIVE = "Paderborn_QML"
NOMBRE_NUEVO_PIPELINE = "NuevoPipeline"

WORK_DIR = "/content" if EN_COLAB else os.getcwd()
TEMP_DIR = os.path.join(WORK_DIR, "temp_paderborn_vibraciones")
os.makedirs(TEMP_DIR, exist_ok=True)

DRIVE_DIR = None
if EN_COLAB:
    from google.colab import drive
    drive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    if not os.path.exists(drive_root):
        drive.mount("/content/drive")
        drive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    
    # Crear carpeta exclusiva del nuevo pipeline al lado de la existente
    DRIVE_DIR = os.path.join(drive_root, CARPETA_RAIZ_DRIVE, NOMBRE_NUEVO_PIPELINE)
    os.makedirs(DRIVE_DIR, exist_ok=True)

# Parámetros físicos del ventaneado
WINDOW_SIZE = 2048              # Tamaño de ventana (~32 ms a 64 kHz)
CANAL_SENAL = "vibration_1"     # Canal de acelerometría mecánica (vibraciones)

# Rutas de salida para los artefactos crudos
OUT_CSV_RAW = os.path.join(WORK_DIR, "paderborn_vibracion_raw.csv")
OUT_CSV_META = os.path.join(WORK_DIR, "paderborn_vibracion_metadatos.csv")
OUT_NPZ_RAW = os.path.join(WORK_DIR, "paderborn_vibracion_raw.npz")

print(f"✅ Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
if DRIVE_DIR:
    print(f"📁 Carpeta de destino en Google Drive: {DRIVE_DIR}")
print(f"🎯 Canal seleccionado: {CANAL_SENAL} | Ventana: {WINDOW_SIZE} muestras (64 kHz)")

In [ ]:
# ==============================================================================
# CELDA 2: CATÁLOGO DE ZENODO Y SELECCIÓN (MÁS BUENOS QUE MALOS)
# ==============================================================================
ZENODO_CATALOG = {
    # --- MOTORES SANOS / BUENOS (Normales, Label = 0) ---
    "K001.rar": "https://zenodo.org/records/15845309/files/K001.rar?download=1",
    "K002.rar": "https://zenodo.org/records/15845309/files/K002.rar?download=1",
    "K003.rar": "https://zenodo.org/records/15845309/files/K003.rar?download=1",
    "K004.rar": "https://zenodo.org/records/15845309/files/K004.rar?download=1",
    "K005.rar": "https://zenodo.org/records/15845309/files/K005.rar?download=1",
    "K006.rar": "https://zenodo.org/records/15845309/files/K006.rar?download=1",

    # --- MOTORES CON FALLAS REALES / MALOS (Fatiga acelerada, Label = 1) ---
    "KA04.rar": "https://zenodo.org/records/15845309/files/KA04.rar?download=1",
    "KA15.rar": "https://zenodo.org/records/15845309/files/KA15.rar?download=1",
    "KI04.rar": "https://zenodo.org/records/15845309/files/KI04.rar?download=1",
    "KI14.rar": "https://zenodo.org/records/15845309/files/KI14.rar?download=1",
    "KB23.rar": "https://zenodo.org/records/15845309/files/KB23.rar?download=1",
    "KB27.rar": "https://zenodo.org/records/15845309/files/KB27.rar?download=1",

    # --- MOTORES CON FALLAS ARTIFICIALES (EDM, Label = 1) ---
    "KA01.rar": "https://zenodo.org/records/15845309/files/KA01.rar?download=1",
    "KI01.rar": "https://zenodo.org/records/15845309/files/KI01.rar?download=1",
}

# Selección por defecto: 4 rodamientos sanos (K001..K004) vs 3 rodamientos con falla real (KA04, KI04, KB23)
# Proporción: ~10,000 ventanas sanas vs ~7,500 de fallas (mayoría sana)
ARCHIVOS_A_DESCARGAR = [
    # 🟢 BUENOS / SANOS (Label = 0)
    "K001.rar",
    "K002.rar",
    "K003.rar",
    "K004.rar",
    # (Descomenta si deseas aún más datos sanos):
    # "K005.rar",
    # "K006.rar",

    # 🔴 MALOS / FALLAS REALES (Label = 1)
    "KA04.rar",  # Pista exterior por fatiga
    "KI04.rar",  # Pista interior por fatiga
    "KB23.rar",  # Falla combinada por fatiga
]

n_buenos = sum(1 for f in ARCHIVOS_A_DESCARGAR if f.startswith("K0"))
n_malos = len(ARCHIVOS_A_DESCARGAR) - n_buenos
print(f"📋 Archivos seleccionados para descarga ({len(ARCHIVOS_A_DESCARGAR)} en total):")
print(f"   • Buenos (Sanos): {n_buenos} -> {[f for f in ARCHIVOS_A_DESCARGAR if f.startswith('K0')]}")
print(f"   • Malos (Fallas): {n_malos} -> {[f for f in ARCHIVOS_A_DESCARGAR if not f.startswith('K0')]}")

In [ ]:
# ==============================================================================
# CELDA 3: DESCARGA STREAMING Y DESCOMPRESIÓN AUTOMÁTICA
# ==============================================================================
def extraer_rar(rar_path, target_folder):
    """Descomprime un archivo .rar usando unrar nativo, tar o rarfile."""
    if shutil.which("unrar"):
        res = subprocess.run(["unrar", "x", "-o+", rar_path, target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "unrar"

    if shutil.which("tar"):
        res = subprocess.run(["tar", "-xf", rar_path, "-C", target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "tar"

    with rarfile.RarFile(rar_path) as rf:
        rf.extractall(path=target_folder)
    return "rarfile"


def download_and_extract_selected(file_list, catalog, target_folder):
    """Descarga con progreso y descomprime los archivos RAR en target_folder."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print(f"\n[1/3] Descargando {len(file_list)} archivos RAR en {target_folder}...")

    for idx, filename in enumerate(file_list, 1):
        if filename not in catalog:
            print(f"  [!] '{filename}' no está en el catálogo. Omitiendo.")
            continue

        rar_path = os.path.join(target_folder, filename)
        if not os.path.exists(rar_path):
            print(f"\n  [{idx}/{len(file_list)}] Descargando {filename} desde Zenodo...")
            req = urllib.request.Request(catalog[filename], headers=headers)
            try:
                with urllib.request.urlopen(req) as resp, open(rar_path, "wb") as out_f:
                    total = int(resp.headers.get("Content-Length", 0))
                    downloaded = 0
                    while True:
                        buf = resp.read(1024 * 1024)
                        if not buf:
                            break
                        downloaded += len(buf)
                        out_f.write(buf)
                        mb_d = downloaded / (1024 * 1024)
                        if total > 0:
                            pct = (downloaded / total) * 100
                            sys.stdout.write(f"\r      [⬇] {mb_d:.1f} / {total/(1024*1024):.1f} MB ({pct:.1f}%)")
                        else:
                            sys.stdout.write(f"\r      [⬇] {mb_d:.1f} MB")
                        sys.stdout.flush()
                print(f"\n      [✓] Descarga completa: {filename}")
            except Exception as e:
                print(f"\n      [X] Error descargando {filename}: {e}")
                if os.path.exists(rar_path):
                    os.remove(rar_path)
                continue
        else:
            print(f"  [{idx}/{len(file_list)}] [✓] Ya descargado: {filename}")

        # Descompresión
        print(f"      [📂] Descomprimiendo {filename}...")
        try:
            metodo = extraer_rar(rar_path, target_folder)
            print(f"      [✓] Extraído exitosamente (método: {metodo})")
        except Exception as e:
            print(f"      [X] Error al descomprimir {filename}: {e}")

In [ ]:
# ==============================================================================
# CELDA 4: EXTRACCIÓN DE SEÑALES DE VIBRACIÓN (VENTANAS + METADATOS)
# ==============================================================================
def parse_metadata_from_filename(file_path):
    """Extrae condiciones operativas y etiqueta física a partir del nombre del .mat."""
    name = os.path.basename(file_path).upper()
    rpm = 900 if "N09" in name else 1500
    torque = 0.1 if "M01" in name else 0.7
    force = 400 if "F04" in name else 1000

    if "K0" in name:
        fault_type, label = "Normal", 0
    elif "KA" in name:
        fault_type, label = "Pista_Exterior", 1
    elif "KI" in name:
        fault_type, label = "Pista_Interior", 1
    elif "KB" in name:
        fault_type, label = "Combinado", 1
    else:
        fault_type, label = "Falla", 1

    return rpm, torque, force, fault_type, label


def extraer_vector_1d(data_obj):
    """Aplana recursivamente estructuras MATLAB a un array numpy 1D float64."""
    arr = data_obj
    while isinstance(arr, np.ndarray) and arr.dtype == object and arr.size > 0:
        arr = arr.flat[0]
    return np.ravel(arr).astype(np.float64)


def get_vibration_channel(mat_path, min_len=200000):
    """Extrae específicamente el canal del acelerómetro / vibración mecánica (64 kHz)."""
    mat = scipy.io.loadmat(mat_path)
    key = next(k for k in mat.keys() if not k.startswith("__"))
    y_channels = mat[key][0, 0]["Y"]
    n_channels = y_channels.shape[1] if y_channels.ndim > 1 else y_channels.shape[0]

    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        ch_name = ""
        if "Name" in ch.dtype.names:
            name_val = ch["Name"]
            while isinstance(name_val, np.ndarray) and name_val.size > 0:
                name_val = name_val.flat[0]
            ch_name = str(name_val).lower()

        signal = extraer_vector_1d(ch["Data"])
        if len(signal) < min_len:
            continue

        if any(w in ch_name for w in ["vib", "acc", "vibration", "beschleunigung"]):
            return signal

    # Fallback si no tiene nombre explícito
    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        signal = extraer_vector_1d(ch["Data"])
        if len(signal) >= min_len:
            return signal

    return None


def process_all_vibration_mats(input_dir, window_size=WINDOW_SIZE, archivos_obj=ARCHIVOS_A_DESCARGAR):
    """
    Extrae las ventanas de vibración cruda (2048 muestras), calcula métricas estadísticas
    de impacto y la FFT completa (1025 bins hasta 32 kHz) sin modificar datos.
    """
    codigos_obj = {os.path.splitext(f)[0].upper() for f in archivos_obj} if archivos_obj else None
    mat_files = []
    for root, _, files in os.walk(input_dir):
        for f in sorted(files):
            if f.lower().endswith(".mat") and not f.startswith("._"):
                if codigos_obj and not any(c in f.upper() for c in codigos_obj):
                    continue
                mat_files.append(os.path.join(root, f))

    print(f"\n[2/3] Procesando {len(mat_files)} archivos .mat (Canal='{CANAL_SENAL}')...")
    
    records = []
    raw_windows_list = []
    fft_full_list = []
    omitidos = 0

    for mat_path in mat_files:
        filename = os.path.basename(mat_path)
        rpm, torque, force, fault_type, label = parse_metadata_from_filename(mat_path)

        try:
            sig = get_vibration_channel(mat_path)
            if sig is None:
                omitidos += 1
                continue

            num_windows = len(sig) // window_size
            for w_idx in range(num_windows):
                seg = sig[w_idx * window_size : (w_idx + 1) * window_size]
                
                # Métricas temporales puras de choque
                rms = float(np.sqrt(np.mean(seg ** 2)))
                std = float(np.std(seg))
                if std > 1e-12:
                    kurt = float(kurtosis(seg, fisher=True, bias=True))
                    skw = float(skew(seg, bias=True))
                else:
                    kurt, skw = 0.0, 0.0
                crest = float(np.max(np.abs(seg)) / (rms + 1e-12))

                # FFT completa pura (1025 puntos desde 0 Hz hasta 32,000 Hz)
                seg_centered = seg - np.mean(seg)
                fft_full = np.abs(np.fft.rfft(seg_centered)).astype(np.float32)

                raw_windows_list.append(seg.astype(np.float32))
                fft_full_list.append(fft_full)

                # Registro tabular
                rec = {
                    "Archivo": filename,
                    "Ventana": w_idx,
                    "Tipo_Falla": fault_type,
                    "RPM": rpm,
                    "Torque_Nm": torque,
                    "Fuerza_N": force,
                    "RMS": rms,
                    "Kurtosis": kurt,
                    "Skewness": skw,
                    "Crest_Factor": crest,
                    "Label": int(label),
                }
                # Guardamos los primeros 64 bins en CSV como referencia inmediata
                for b_idx in range(64):
                    rec[f"fft_{b_idx}"] = float(fft_full[b_idx])

                records.append(rec)

        except Exception as e:
            print(f"  [X] Error procesando {filename}: {e}")

    print(f"  [✓] Ventanas extraídas: {len(records):,}")
    if omitidos > 0:
        print(f"  [!] Archivos omitidos (sin canal): {omitidos}")

    df_tabular = pd.DataFrame(records)
    arr_windows = np.array(raw_windows_list, dtype=np.float32)
    arr_fft_full = np.array(fft_full_list, dtype=np.float32)

    return df_tabular, arr_windows, arr_fft_full

In [ ]:
# ==============================================================================
# CELDA 5: EJECUCIÓN, VALIDACIÓN Y EXPORTACIÓN LIMPIA A GOOGLE DRIVE
# ==============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print(" 🚀 INICIANDO ETL: EXTRACCIÓN CRUDA DE VIBRACIONES (PADERBORN) ")
    print("=" * 70)

    # 1. Descarga y descompresión
    download_and_extract_selected(ARCHIVOS_A_DESCARGAR, ZENODO_CATALOG, TEMP_DIR)

    # 2. Extracción de señales de vibración cruda
    df_tabular, arr_windows, arr_fft_full = process_all_vibration_mats(TEMP_DIR, window_size=WINDOW_SIZE)

    if len(df_tabular) == 0:
        raise RuntimeError("No se extrajeron registros. Revisa ARCHIVOS_A_DESCARGAR.")

    # 3. Comprobación de integridad
    n_nulos = df_tabular.isnull().sum().sum()
    print(f"\n[3/3] Verificación de Calidad Física:")
    print(f"   • Total de ventanas:       {len(df_tabular):,}")
    print(f"   • Matriz temporal en disco: {arr_windows.shape} (2048 muestras a 64 kHz)")
    print(f"   • Matriz FFT completa:     {arr_fft_full.shape} (1025 bins hasta 32 kHz)")
    print(f"   • Valores nulos en tabla:  {n_nulos}")

    # 4. Guardado en disco local
    df_tabular.to_csv(OUT_CSV_RAW, index=False)
    
    # Separamos metadatos ligeros para consultas rápidas
    cols_meta = ["Archivo", "Ventana", "Tipo_Falla", "RPM", "Torque_Nm", "Fuerza_N", "Label"]
    df_tabular[cols_meta].to_csv(OUT_CSV_META, index=False)

    # Guardado del archivo .npz compacto con los datos crudos completos
    np.savez_compressed(
        OUT_NPZ_RAW,
        ventanas_tiempo=arr_windows,              # (N, 2048) señal temporal cruda
        fft_completa=arr_fft_full,                # (N, 1025) magnitud FFT pura de 0 a 32 kHz
        labels=df_tabular["Label"].values,        # (N,) 0 = sano, 1 = falla real
        rpm=df_tabular["RPM"].values,
        torque=df_tabular["Torque_Nm"].values,
        fuerza=df_tabular["Fuerza_N"].values,
        tipo_falla=df_tabular["Tipo_Falla"].values,
        archivos=df_tabular["Archivo"].values,
        stats_temporales=df_tabular[["RMS", "Kurtosis", "Skewness", "Crest_Factor"]].values,
    )

    print(f"\n💾 Archivos crudos listos en entorno de trabajo:")
    print(f"   • Dataset tabular (.csv):    {OUT_CSV_RAW}")
    print(f"   • Metadatos (.csv):          {OUT_CSV_META}")
    print(f"   • Dataset completo (.npz):   {OUT_NPZ_RAW} ({os.path.getsize(OUT_NPZ_RAW)/(1024*1024):.1f} MB)")

    # 5. Copia directa a la carpeta principal en Google Drive
    if DRIVE_DIR is not None:
        print(f"\n📁 Copiando a Google Drive ({DRIVE_DIR})...")
        for r_local in (OUT_CSV_RAW, OUT_CSV_META, OUT_NPZ_RAW):
            fname = os.path.basename(r_local)
            shutil.copy2(r_local, os.path.join(DRIVE_DIR, fname))
            print(f"   [✓] Copiado: {fname}")
        print(f"\n🎉 DATASET CRUDO COMPLETAMENTE DISPONIBLE EN GOOGLE DRIVE:")
        print(f"   📂 {DRIVE_DIR}")

    # 6. Resumen de balance de clases
    print("\n" + "=" * 70)
    print(" RESUMEN DEL DATASET CRUDO (BALANCE DE CLASES)")
    print("=" * 70)
    conteo = df_tabular["Label"].value_counts()
    n_sanos = conteo.get(0, 0)
    n_fallas = conteo.get(1, 0)
    print(f"  • Motores Buenos (Sanos, Label=0): {n_sanos:,} ventanas ({(n_sanos/len(df_tabular))*100:.1f}%)")
    print(f"  • Motores Malos (Fallas, Label=1): {n_fallas:,} ventanas ({(n_fallas/len(df_tabular))*100:.1f}%)")
    print(f"  • Total Ventanas Extraídas:        {len(df_tabular):,}")

    print("\nDesglose por Tipo de Rodamiento:")
    print(df_tabular["Tipo_Falla"].value_counts().to_string())

    print("\nDesglose por RPM y Torque:")
    print(pd.crosstab(df_tabular["RPM"], df_tabular["Torque_Nm"], margins=True))

    print("\nMedianas de Métricas Temporales Crudas (Sano vs Falla):")
    print(df_tabular.groupby("Label")[["RMS", "Kurtosis", "Skewness", "Crest_Factor"]].median().round(4).to_string())